# Milestone 2 — hh-rlhf data pipeline (v1.1)

Run the cells **top to bottom**. Each pipeline step is a separate script in `src/`; its docstring states the input, output and rules.

**Before you start** (one-off): add a Colab Secret — key icon 🔑 in the left sidebar → *Add new secret* → Name `DHAI_BUCKET`, Value = your bucket name (e.g. `bc-rlhf-reward-2026`) → switch on *Notebook access*.

## 0. Settings — edit these two lines

In [ ]:
# Settings used by the cells below.
GITHUB_REPO = "bounchun/rlhf-reward-model-pipeline-m2"   # <- your GitHub user/repo
GCP_PROJECT = "your-gcp-project-id"              # <- Google Cloud console, project picker at the top

In [ ]:
# Download the code from GitHub and install the pinned libraries (exact versions from requirements.txt).
!git clone https://github.com/{GITHUB_REPO}.git repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# Log in to Google Cloud (no key files; also used by the gcloud CLI below)
from google.colab import auth, userdata
auth.authenticate_user()
!gcloud config set project {GCP_PROJECT}

# Read the bucket name from Colab Secrets, so it is never written in the code.
import os
os.environ['DHAI_BUCKET'] = userdata.get('DHAI_BUCKET')   # bucket name from Colab Secrets
print('Bucket:', os.environ['DHAI_BUCKET'])

## 0b. Create the buckets — ONE-OFF, skip if they already exist
Creates `<bucket>` (Standard) and `<bucket>-holdout` (Coldline) in europe-west1 with versioning, lifecycle rules and two service accounts.

In [ ]:
# One-off: create the main bucket (Standard) and the holdout bucket (Coldline),
# with versioning, clean-up rules and the two service accounts. Skip if they already exist.
!PROJECT={GCP_PROJECT} BUCKET={os.environ['DHAI_BUCKET']} bash scripts/setup_gcs.sh

## Optional: check everything offline first
Tests + a full demo on synthetic data (no GCS, no Hugging Face). Should end with `OK`.

In [ ]:
# Offline check (no Google Cloud, no Hugging Face):
# 1) the 43 unit tests, which should end with "OK";
# 2) the whole pipeline on fake data, which should print "demo OK".
!make test 2>&1 | tail -3
!DHAI_BUCKET= make demo > /dev/null && make clean-demo && echo 'demo OK'

## Pin the dataset version (skip: already pinned)
`config.json` is already pinned to commit `09be8c5bbc57cb3887f3a9732ad6aa7ec602a1fa`, so **skip this cell** for a normal run.
Only run it to move to a newer revision of the dataset: it rewrites `config.json`, so the same change must then be committed to GitHub, or the manifest will record a `-dirty` commit.

In [ ]:
# SKIP for a normal run: config.json is already pinned.
# Only for moving to a newer dataset version: ask Hugging Face for the latest commit hash
# and write it into config.json (then commit the same change to GitHub).
import json, subprocess
rev = subprocess.check_output(['python', '-m', 'src.step1_scrape_raw', '--resolve-revision'], text=True).strip()
cfg = json.load(open('config.json'))
cfg['dataset']['hf_revision'] = rev
json.dump(cfg, open('config.json', 'w'), indent=2, ensure_ascii=False)
print('Pinned hf_revision =', rev)

## Step 1 — scrape the raw data
Downloads the 8 raw files at the pinned revision and records their SHA-256 checksums.

In [ ]:
# Step 1: download the 8 raw files at the pinned commit and save their SHA-256 checksums.
# Output: data/downloads/<commit>/ and manifests/raw-<commit>.json
!python -m src.step1_scrape_raw

## Step 2 — move the raw data into storage
Uploads to `gs://<bucket>/raw/hh-rlhf/<revision>/` (write-once, read-back verified).

In [ ]:
# Step 2: upload the raw files to gs://<bucket>/raw/, read them back to check them.
# raw/ is write-once: a file with different content is never overwritten.
!python -m src.step2_store_raw

## Step 3 — validate and clean

In [ ]:
# Step 3: check the raw checksums again, then apply the 5 cleaning rules.
# Prints how many rows each rule dropped. Output: data/interim/<version>/cleaned.parquet
!python -m src.step3_clean

## Step 4 — quality checks (poisoning screen, PII, lengths)

In [ ]:
# Step 4: hard checks (the run stops if one fails), poisoning screen, PII scan, length summary.
# Output: gs://<bucket>/audit/<version>/
!python -m src.step4_quality_checks

## Step 5 — extract features

In [ ]:
# Step 5: add group_id, swap A/B to balance the labels, and add the length and refusal columns.
# Output: data/interim/<version>/features.parquet
!python -m src.step5_features

## Step 6 — reserve future data, filter to the working set

In [ ]:
# Step 6: set aside the future data (F1, F2), then take the ~30k working set in whole groups.
!python -m src.step6_filter

## Step 7 — train / dev / test split

In [ ]:
# Step 7: split the working set into train/dev/test (80/10/10) by conversation group,
# check that no group is in two splits, and fit the length-outlier threshold on train only.
!python -m src.step7_split

## Step 8 — sharding (off by default)

In [ ]:
# Step 8: optional training shards. Off by default (num_shards = 1 in config.json),
# so this only prints that it was skipped.
!python -m src.step8_shard

## Step 9 — store the preprocessed data

In [ ]:
# Step 9: final checks, then upload train/dev to gs://<bucket>/processed/<version>/
# and test/F1/F2 to the holdout bucket, with their checksums in the manifest.
!python -m src.step9_store_processed

## Step 10 — blind human-review sample

In [ ]:
# Step 10: export 200 train pairs, without labels, for checking the labels by hand.
# Output: audit/label_review_v1.1.csv (stays local, never committed to Git)
!python -m src.step10_review_sample

## Step 11 — model-ready inputs (Milestone 2)
Text inputs with left-truncated contexts and log-length features standardised with the **train** mean/SD. Train/dev go to `processed/<version>/model_inputs/`, test/F1/F2 to the holdout bucket.

In [ ]:
# Step 11: build model-ready inputs; the scaler is fitted on train only and applied to every split.
!python -m src.step11_model_inputs

## Reconciliation — a second, independent method (Milestone 2)
Recounts the raw files with plain Python and compares them with what the pipeline stored. Should end with `All 7 checks agree`.

In [ ]:
# Independent recount: raw rows, row accounting, checksums, label direction, split overlap.
!python -m src.reconcile

## EDA (Milestone 2)
Writes 9 figures, `eda_summary.json` and `findings.csv` to `docs/eda/`. Distributions use **train only**; the holdout bucket is never read. Takes a few minutes (the reply-similarity step).

In [ ]:
# EDA on the real data. Outputs: docs/eda/fig1..fig9, eda_summary.json, findings.csv
!python -m src.eda
import pandas as pd
pd.set_option('display.max_colwidth', 200)
pd.read_csv('docs/eda/findings.csv')

In [ ]:
# Download the EDA outputs and the manifest, to commit them to GitHub (they hold no dataset text).
import shutil
from google.colab import files
shutil.make_archive('m2_outputs', 'zip', '.', 'docs/eda')
files.download('m2_outputs.zip')
files.download('manifests/v1.1.json')

## Sample queries
Example pandas queries on the stored train/dev data (also saved to `audit/<version>/sample_queries.json`).

In [ ]:
# Example pandas queries on the stored train/dev files (reads them from the bucket).
# --show 2 also prints 2 example pairs. Output: gs://<bucket>/audit/<version>/sample_queries.json
!python -m src.sample_queries --show 2

## Results at a glance

In [ ]:
# Summary of the run, read from the manifest: rows dropped, poisoning screen, split sizes.
import json
import pandas as pd
m = json.load(open('manifests/v1.1.json'))['steps']
print('Rows dropped:', json.dumps(m['step3_clean']['rows_dropped'], indent=2))
print('Poison screen:', m['step4_quality_checks']['poison_screen'])
pd.DataFrame(m['step7_split']['splits']).T[['pairs', 'groups', 'label_1_share', 'long_outlier_share']]

## Verify the buckets and read a file back

A last check, like Cell 8 of the GCS lab: list every file in the two buckets, then read `dev.parquet` straight from the bucket with pandas. Only a few non-text columns are shown, because the dataset contains harmful text.

In [ ]:
# Final check, like Cell 8 of the GCS lab:
# 1) list every file in the main bucket and the holdout bucket, with its size;
# 2) read dev.parquet straight from the bucket with pandas (gcsfs handles gs:// paths).
# Text columns are not printed: the dataset contains harmful text.
import fsspec
import pandas as pd
from src.common import load_config, storage_roots, join

cfg = load_config()
version = cfg['dataset']['version']
main_root, holdout_root = storage_roots(cfg)   # gs://<bucket> and gs://<bucket>-holdout

# 1. List the files in both buckets
for root in [main_root, holdout_root]:
    fs, path = fsspec.core.url_to_fs(root)
    print(f'\n--- Files in {root} ---')
    for name, info in sorted(fs.find(path, detail=True).items()):
        print(f'  - {name} ({info["size"]:,} bytes)')

# 2. Read dev.parquet directly from the bucket
dev_path = join(main_root, 'processed', version, 'dev.parquet')
dev_df = pd.read_parquet(dev_path)
print(f'\n--- Read test ({dev_path}) ---')
print(f'Rows loaded: {len(dev_df):,}')   # 2,876 (same as v1.0)
display(dev_df[['pair_id', 'subset', 'num_turns', 'len_diff', 'label']].head(3))

## Milestone 1 only: fill the README (do NOT run for Milestone 2)
This cell belongs to the Milestone 1 workflow: it fills the M1 README from `manifests/v1.0.json`. The Milestone 2 README is already final, so running it here would only produce an M1-style README with v1.0 numbers.

In [ ]:
# DO NOT RUN for Milestone 2 (Milestone 1 helper, reads manifests/v1.0.json).
# SKIP: the README on GitHub is already final.
# Fills the README numbers from the manifest and downloads the files to upload to GitHub.
import json
from google.colab import files
rev = json.load(open('config.json'))['dataset']['hf_revision']

!python scripts/fill_readme.py --bucket {os.environ['DHAI_BUCKET']} --repo {GITHUB_REPO.split('/')[1]} --reviewers screenshots

files.download('README.md')
files.download('manifests/v1.1.json')
files.download(f'manifests/raw-{rev}.json')

## Human review (can be done later)
Download the sheet, fill `my_choice` with A / B / tie, upload it back to `repo/audit/` with the Colab file browser, then run the scoring cell.
⚠️ Do not put this CSV on GitHub: it contains dataset text (it is git-ignored).

In [ ]:
# Download the review sheet to fill in by hand (my_choice = A / B / tie).
# It contains dataset text: keep it on your computer, never on GitHub.
from google.colab import files
files.download('audit/label_review_v1.1.csv')

In [ ]:
# After uploading the filled sheet back to audit/: score it against the crowdworkers' labels
# and download the updated manifest.
from google.colab import files
!python -m src.score_review
files.download('manifests/v1.1.json')